In [ ]:
import matplotlib.pyplot as plt 
import numpy as np
import glob
from ZTF_Pipeline import SingleFrame, PipelineConfig, MaskConfig, BackgroundConfig

Field = "664"
Filter = "zr"
CCD = "10"
Quadrant = "4"


FCQZ = f'{Field}_c{CCD}_q{Quadrant}_{Filter}'

# Chemin vers le fichier FITS
fits_folder = sorted(glob.glob(f".\\ztf_data\\{FCQZ}\\sciimg\\*.fits"))
fits_file = fits_folder[0]

# Initialisation d'une config par défaut
config = PipelineConfig()
frame = SingleFrame(fits_file, config)


frame.summary()

In [ ]:
import matplotlib.patches as patches

def radec_to_pixel(ra, dec, wcs, integer=False):
    skycoord = wcs.world_to_pixel_values(ra, dec)
    x = skycoord[0]  
    y = skycoord[1]  
    if integer:
        x, y = int(x), int(y)
    return x, y

def Mask_Test(frame, sigma1, dilate1, sigma2, dilate2, center_pix=None, vmin=None, vmax=None):
    """
    show the effect of different mask parameters on the same image, with a zoom on a specific area (center_pix) if provided.
    """ 
    orig_sigma = frame.cfg.mask.detect_sigma
    orig_dilate = frame.cfg.mask.dilate_pix

    frame.cfg.mask.detect_sigma = sigma1
    frame.cfg.mask.dilate_pix = dilate1
    mask1 = frame.build_mask()
    
    frame.cfg.mask.detect_sigma = sigma2
    frame.cfg.mask.dilate_pix = dilate2
    mask2 = frame.build_mask()

    frame.cfg.mask.detect_sigma = orig_sigma
    frame.cfg.mask.dilate_pix = orig_dilate


    h, w = frame.data.shape
    if center_pix is None:
        cx, cy = w // 2, h // 2
    else:
        cx, cy = center_pix

    z_size = 100
    y_min, y_max = int(np.clip(cy - z_size, 0, h)), int(np.clip(cy + z_size, 0, h))
    x_min, x_max = int(np.clip(cx - z_size, 0, w)), int(np.clip(cx + z_size, 0, w))
    zoom_slice = (slice(y_min, y_max), slice(x_min, x_max))

    
    vmin = vmin if vmin is not None else np.percentile(frame.data, 5) 
    vmax = vmax if vmax is not None else np.percentile(frame.data, 98)
    
    plt.style.use('dark_background')
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))

    col_titles = ["Image", f"Mask A (σ={sigma1})", f"Mask B (σ={sigma2})"]
    datasets = [frame.data, mask1, mask2]
    cmaps = ['viridis', 'gray', 'gray']

    for i in range(3):
        # Complete images
        ax_full = axes[0, i]
        img = ax_full.imshow(datasets[i], cmap=cmaps[i], origin='lower', 
                             vmin=(vmin if i==0 else None), vmax=(vmax if i==0 else None))
        ax_full.set_title(col_titles[i])
        
        # Rectangle of zoom area
        rect = patches.Rectangle((x_min, y_min), x_max-x_min, y_max-y_min, 
                                 linewidth=2, edgecolor='red', facecolor='none')
        ax_full.add_patch(rect)
        
        if i == 0:
            fig.colorbar(img, ax=ax_full, label='ADU', fraction=0.046, pad=0.04)

        # Zooms
        ax_zoom = axes[1, i]
        ax_zoom.imshow(datasets[i][zoom_slice], cmap=cmaps[i], origin='lower',
                        vmin=(vmin if i==0 else None), vmax=(vmax if i==0 else None))
        ax_zoom.set_title(f"Zoom ({cx:.0f}, {cy:.0f})")

    plt.tight_layout()
    plt.show()

#TEST ME
ra = 127.448018
dec = 33.906536
x, y = radec_to_pixel(ra, dec, frame.wcs)

Mask_Test(frame, 3.0, 6, 3.0, 8, center_pix=(x, y), vmin=np.nanmedian(frame.data), vmax=np.nanmedian(frame.data) + 20)

In [ ]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from ipywidgets import interact, FloatSlider

# --- PRÉPARATION DES DONNÉES ---
ra = 127.448018
dec = 33.906536
x, y = radec_to_pixel(ra, dec, frame.wcs)

h, w = frame.data.shape
size = 400
cx, cy = int(x), int(y) 
sub_data = frame.data[cy-size:cy+size, cx-size:cx+size]

# Création de la grille de coordonnées pour le plot 3D
x = np.arange(sub_data.shape[1])
y = np.arange(sub_data.shape[0])
X, Y = np.meshgrid(x, y)

def plot_3d_threshold(smooth_sigma, detect_sigma):
    # 1. Lissage (Smoothing)
    img_s = gaussian_filter(sub_data, sigma=smooth_sigma)
    
    # 2. Calcul du seuil (Threshold)
    med = np.nanmedian(img_s)
    sig = frame._robust_sigma(img_s)
    thresh_value = med + detect_sigma * sig
    
    # --- VISUALIZATION ---
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection='3d')
    # Surface smoothed
    surf = ax.plot_surface(X, Y, img_s, cmap='viridis', alpha=0.8, antialiased=True)
    # Treshold plane
    Z_thresh = np.full_like(img_s, thresh_value)
    ax.plot_surface(X, Y, Z_thresh, color='red', alpha=0.3)
    
    ax.set_zlabel('Flux (ADU)')
    ax.set_title(f'Visualization treshold detection:\nSmoothing Sigma: {smooth_sigma} | Treshold: {detect_sigma}σ')
    fig.colorbar(surf, ax=ax, shrink=0.5, aspect=10, label='Flux lissé')
    
    plt.show()

# --- INTERFACE INTERACTIVE ---
interact(
    plot_3d_threshold, 
    smooth_sigma=FloatSlider(min=0.1, max=5.0, step=0.1, value=1.5, description='smooth_sigma (px)'),
    detect_sigma=FloatSlider(min=0.5, max=10.0, step=0.5, value=3.0, description='detect_sigma (N-sig)')
)

In [ ]:
# --- CONFIGURATION ---
box_sizes = [4, 8, 16, 32, 48, 64, 128]
n_tests = len(box_sizes)

fig = plt.figure(figsize=(20, 12))
gs = fig.add_gridspec(3, n_tests)

# Orignial Image 
ax_orig = fig.add_subplot(gs[0, :])
vmin, vmax = np.nanmedian(frame.data), np.nanmedian(frame.data) + 3
im_orig = ax_orig.imshow(frame.data, origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
ax_orig.set_title("Original Image (Reference)", fontsize=14, fontweight='bold')
plt.colorbar(im_orig, ax=ax_orig, pad=0.01)

# Loop over box sizes
for i, size in enumerate(box_sizes):
    # Update config and compute
    frame.cfg.bkg.box_size = size
    m = frame.build_mask()
    bkg_map = frame.estimate_background(mask=m)
    data_sub = frame.data - bkg_map
    
    # Bkg Map
    ax_bkg = fig.add_subplot(gs[1, i])
    im_bkg = ax_bkg.imshow(bkg_map, origin='lower', cmap='plasma')
    ax_bkg.set_title(f"BKG Map (Box: {size})")
    plt.colorbar(im_bkg, ax=ax_bkg, orientation='horizontal', pad=0.1)
    
    # Subtracted Image with same vmin/vmax as original for fair comparison
    ax_sub = fig.add_subplot(gs[2, i])
    im_sub = ax_sub.imshow(data_sub, origin='lower', cmap='viridis', vmin=np.nanmedian(data_sub), vmax=np.nanmedian(data_sub) + 3)
    ax_sub.set_title(f"Subtracted (Box: {size})")
    plt.colorbar(im_sub, ax=ax_sub, orientation='horizontal', pad=0.1)

plt.tight_layout()
plt.show()

In [ ]:
import copy

# Test configuration
seeing_target = frame.seeing + 2.0 

frame_test_seeing = copy.deepcopy(frame)
frame_test_seeing.psf_homogenize_to(seeing_target)

# Zoom on one star
ra, dec = 127.448018, 33.906536
x, y = radec_to_pixel(ra, dec, frame.wcs, integer=True)
z = (slice(y-50, y+50), slice(x-50, x+50))

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
vmin, vmax = np.nanpercentile(frame.data[z], [1, 99])

# --- Original Image ---
ax[0].imshow(frame.data[z], origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
ax[0].set_title(f"Original (Seeing: {frame.seeing:.2f}\")")

# --- After PSF Homogenization (using the copy object) ---
ax[1].imshow(frame_test_seeing.data[z], origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
ax[1].set_title(f"Homogenize (Target: {seeing_target:.2f}\")")


plt.colorbar(ax[1].get_images()[0], ax=ax, label='Flux (ADU)')

plt.show()

In [6]:
from ZTF_Pipeline import ZTFFolderPipeline, PipelineConfig
import matplotlib.pyplot as plt
import numpy as np

Field = "664"
Filter = "zr"
CCD = "10"
Quadrant = "4"
FCQZ = f'{Field}_c{CCD}_q{Quadrant}_{Filter}'
folder_path = f".\\ztf_data\\{FCQZ}\\sciimg"
ref_path = f".\\ztf_data\\{FCQZ}\\refimg\\{FCQZ}_refimg.fits"
config = PipelineConfig() 

# 2. Initialize pipeline
pipeline = ZTFFolderPipeline(folder=folder_path, config=config)

# 3. Define the Master Geometry
# We pick the first file as the geometric reference (WCS)
pipeline.set_target_from_file(pipeline.files[0])

print(f"Nomber folder found : {len(pipeline.files)}")
print(f"Resolution : {pipeline.target_shape}")

Nomber folder found : 631
Resolution : (3080, 3072)


In [ ]:
stats = pipeline.plot_seeing_hist(folder=folder_path, pattern="*.fits", bins=40)
stats

In [ ]:
import random
random.seed(1)

def test_reference_creation(target_seeing, target_zp, max_frames=10, save_path=None):
    print(f"\n--- Testing Stack with Seeing Target: {target_seeing}\" ---")
    
    try:
        # reference frame creation 
        ref_frame = pipeline.build_reference(
            zp_target=target_zp,
            seeing_target=target_seeing,
            save_path=save_path,
            max_frames=max_frames, 
            show_ref=False 
        )
        
        fig, ax = plt.subplots(1, 2, figsize=(16, 7))
        vmin, vmax = np.nanpercentile(ref_frame.data, [1, 99])
        
        im0 = ax[0].imshow(ref_frame.data, origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
        ax[0].set_title(f"Reference Stack (Seeing < {target_seeing}\")")
        fig.colorbar(im0, ax=ax[0], label='Flux (ADU)', fraction=0.046, pad=0.04)
        
        h, w = ref_frame.data.shape
        z = (slice(h//2-150, h//2+150), slice(w//2-150, w//2+150))
        im1 = ax[1].imshow(ref_frame.data[z], origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
        ax[1].set_title("ZOOM")
        fig.colorbar(im1, ax=ax[1], label='Flux (ADU)', fraction=0.046, pad=0.04)
        
        plt.tight_layout()
        plt.show()
        
    except ValueError as e:
        print(f"Erreur : {e}")

# --- TEST ZONE ---
pipeline.cfg.mask.detect_sigma = 5.0 
pipeline.cfg.mask.dilate_pix = 8
pipeline.cfg.bkg.box_size = 16

#test_reference_creation(target_seeing=2.0, target_zp=30.0, max_frames=10)
test_reference_creation(target_seeing=2.0, target_zp=30.0, max_frames=42, save_path=ref_path)

In [ ]:
from ZTF_Pipeline import ZTFDifferencePipeline, SingleFrame
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits


ref_frame = SingleFrame(ref_path, pipeline.cfg)
diff_engine = ZTFDifferencePipeline(pipeline, ref_frame)

def plot_subtraction_results(start_date, end_date, n_show=3, zoom_center=None, zoom_size=100, force=False):
    """
    Réalise la soustraction et affiche un tableau de bord complet pour chaque image.
    Utilise les propriétés de SingleFrame pour éviter les accès disques redondants.
    """
    # Récupération des frames (calcule les manquantes ou charge le cache)
    diff_frames = diff_engine.subtract_range(start_date, end_date, save=True, force=force)
    
    if not diff_frames:
        print("Aucune donnée trouvée pour cette période.")
        return

    for _ , diff_f in enumerate(diff_frames[:n_show]):
        # Préparation de la science matchée (pour comparaison visuelle juste)
        sci_f = pipeline.prepare_frame(diff_f.path, zp_target=ref_frame.zp, seeing_target=ref_frame.seeing)
        
        fig, axes = plt.subplots(2, 3, figsize=(18, 10), facecolor='#111111')
        fig.suptitle(f"SOUSTRACTION ZTF | {sci_f.date_str} | {sci_f.path.name}", 
                     fontsize=16, fontweight='bold', color='white', y=0.98)

        # 1. Préparation des données et titres
        titles = [
            f"SCIENCE (Matchée)\nSeeing Original: {sci_f.seeing:.2f}\"",
            f"RÉFÉRENCE (Stack)\nSeeing Target: {ref_frame.seeing:.2f}\"",
            f"DIFFÉRENCE\nZP: {sci_f.zp:.2f}"
        ]
        data_list = [sci_f.data, ref_frame.data, diff_f.data]
        
        # 2. Définition du Zoom
        h, w = sci_f.data.shape
        cx, cy = zoom_center if zoom_center else (w // 2, h // 2)
        z_slice = (slice(max(0, int(cy - zoom_size)), min(h, int(cy + zoom_size))), 
                   slice(max(0, int(cx - zoom_size)), min(w, int(cx + zoom_size))))

        # 3. Calcul des échelles de contraste (basées sur la science pour cohérence)
        vmin, vmax = np.nanpercentile(sci_f.data, [1, 98])
        std_diff = np.nanstd(diff_f.data)

        for col in range(3):
            # Paramètres spécifiques à la colonne Différence
            is_diff = (col == 2)
            cmap = 'bwr' if is_diff else 'viridis' # Bleu-Blanc-Rouge pour la diff
            v_min = -5 * std_diff if is_diff else vmin
            v_max = 5 * std_diff if is_diff else vmax
            
            # --- LIGNE 1 : IMAGE ENTIÈRE ---
            ax_full = axes[0, col]
            im = ax_full.imshow(data_list[col], origin='lower', cmap=cmap, vmin=v_min, vmax=v_max)
            ax_full.set_title(titles[col], color='white', fontsize=12)
            ax_full.axis('off')
            
            # Rectangle de zone zoomée
            rect = plt.Rectangle((cx-zoom_size, cy-zoom_size), zoom_size*2, zoom_size*2, 
                                 edgecolor='red', facecolor='none', lw=1.5, ls='--')
            ax_full.add_patch(rect)
            
            # Colorbar fine
            cbar = fig.colorbar(im, ax=ax_full, fraction=0.046, pad=0.04)
            cbar.ax.tick_params(colors='white')

            # --- LIGNE 2 : ZOOM ---
            ax_zoom = axes[1, col]
            ax_zoom.imshow(data_list[col][z_slice], origin='lower', cmap=cmap, vmin=v_min, vmax=v_max)
            ax_zoom.set_title(f"Zoom {['Science', 'Ref', 'Diff'][col]}", color='gray', fontsize=10)
            ax_zoom.grid(color='white', alpha=0.1, linestyle=':')
            
        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        plt.show()

# --- TEST ---
# Exemple de plage de dates (à adapter selon tes données)
# Si tu veux tester sur tes premières images :
start = "2020-01-01"
end = "2020-04-15"

ra = 127.448018
dec = 33.906536
x, y = radec_to_pixel(ra, dec, frame.wcs)

plot_subtraction_results(start_date=start, end_date=end, n_show=30, zoom_center=(x, y), zoom_size=64, force=True)